# Converting between formats

Every format has a parser into the morphnet `Netlist` and a writer out of it,
so converting a netlist is one line: **parse** with one format, **write**
with another. This page:

1. walks through conversions between pairs of formats, with full output;
2. converts one circuit through all 7 × 7 format pairs and checks that
   connectivity and parameters survive;
3. round-trips a photonic MZI through every format, checking that its
   waveguide nets stay waveguide;
4. goes from a `kfnetlist.Netlist` to every format and back.

The example netlists of each format are described on
[Parsing netlist formats](netlist_formats.py).

In [1]:
from kfnetlist import Netlist, NetlistPort, PortRef, morphnet
from kfnetlist.morphnet import netlist as formats

FORMATS = [
    "spice",
    "hspice",
    "xyce",
    "spectre",
    "spectre_spice",
    "vams",
    "systemverilog",
]


def convert(text: str, src: str, dst: str) -> str:
    """Convert netlist text from format `src` to format `dst`."""
    parse = getattr(formats, f"parse_{src}")
    write = getattr(formats, f"write_{dst}")
    return write(parse(text))

The same helper is a ten-line command-line converter
(`docs/assets/convert_demo.py`): `python convert_demo.py spice spectre < in.sp`.

## Pairwise conversions

### SPICE → Spectre

SPICE primitives (`M`, `R`, `C`, ...) become Spectre instances of the model
or primitive; subcircuit calls keep their pin order.

In [2]:
SPICE = """\
* CMOS buffer
.subckt inverter in out vdd vss
M1 out in vdd vdd pch W=2u L=180n
M2 out in vss vss nch W=1u L=180n
.ends inverter
.subckt buffer a y vdd vss
X1 a mid vdd vss inverter
X2 mid y vdd vss inverter
C1 y vss 10f
.ends buffer
.end
"""
print(convert(SPICE, "spice", "spectre"))

subckt inverter (in out vdd vss)
M1 (out in vdd vdd) pch W=2u L=180n
M2 (out in vss vss) nch W=1u L=180n
ends inverter
subckt buffer (a y vdd vss)
X1 (a mid vdd vss) inverter
X2 (mid y vdd vss) inverter
C1 (y vss) capacitor value=10f
ends buffer



### Spectre → Verilog-AMS

Positional Spectre terminals become named Verilog connections, using the pin
names of the subcircuit or primitive (`d`, `g`, `s`, `b` for a MOSFET).

In [3]:
SPECTRE = """\
subckt rc_lowpass (in_ out gnd)
R1 (in_ out) resistor r=1k
C1 (out gnd) capacitor c=100p
ends rc_lowpass
"""
print(convert(SPECTRE, "spectre", "vams"))

module rc_lowpass (in_, out, gnd);
  inout in_, out, gnd;
  resistor #(.r(1k)) R1 (.p(in_), .n(out));
  capacitor #(.c(100p)) C1 (.p(out), .n(gnd));
endmodule



### Verilog-AMS → SystemVerilog

SystemVerilog has no SI suffixes, so `10k` is written as `10000`. The
Verilog-AMS discipline (`electrical`) is kept as the port's net type.

In [4]:
VAMS = """\
module divider (vin, vout, gnd);
  input vin;
  output vout;
  inout gnd;
  electrical vin, vout, gnd;
  resistor #(.r(10k)) R1 (.p(vin), .n(vout));
  resistor #(.r(10k)) R2 (.p(vout), .n(gnd));
endmodule
"""
print(convert(VAMS, "vams", "systemverilog"))

module divider (
  input electrical vin,
  output electrical vout,
  inout electrical gnd
);
  resistor #(.r(10000)) R1 (.p(vin), .n(vout));
  resistor #(.r(10000)) R2 (.p(vout), .n(gnd));
endmodule



### SystemVerilog → SPICE

Named connections are put back in definition order: modules defined in the
same file become `.subckt` blocks and built-in devices (`mosfet`,
`resistor`, ...) become SPICE primitives with their fixed pin order.
Behavioral code (`always`, `assign` with logic) is not structure and is not
written to SPICE.

SPICE picks the element type from the first letter of the instance name, so
subcircuit instances must be named `X...` and devices `M...`, `R...`, ...
for the SPICE output to parse back.

In [5]:
SYSTEMVERILOG = """\
module inv (input a, output y, inout vdd, vss);
  mosfet #(.W(2e-6), .L(180e-9)) MP (.d(y), .g(a), .s(vdd), .b(vdd));
  mosfet #(.W(1e-6), .L(180e-9)) MN (.d(y), .g(a), .s(vss), .b(vss));
endmodule
module ring3 (output osc, inout vdd, vss);
  inv X1 (.a(osc), .y(n1), .*);
  inv X2 (.a(n1), .y(n2), .*);
  inv X3 (.a(n2), .y(osc), .*);
  always @(posedge osc) $display("tick");
endmodule
"""
print(convert(SYSTEMVERILOG, "systemverilog", "spice"))

* untitled
.subckt inv a y vdd vss
MP y a vdd vdd W=2e-06 L=1.8e-07
MN y a vss vss W=1e-06 L=1.8e-07
.ends inv
.subckt ring3 osc vdd vss
X1 osc n1 vdd vss inv
X2 n1 n2 vdd vss inv
X3 n2 osc vdd vss inv
.ends ring3
.end



### HSPICE → Xyce

In [6]:
HSPICE = """\
* RC low-pass
.subckt rc_lowpass in_ out gnd
R1 in_ out 1k     $ series resistor
C1 out gnd 100p   $ shunt capacitor
.ends rc_lowpass
.end
"""
print(convert(HSPICE, "hspice", "xyce"))

* untitled
.subckt rc_lowpass in_ out gnd
R1 in_ out 1k
C1 out gnd 100p
.ends rc_lowpass
.end



### Mixed Spectre / SPICE → SystemVerilog

A deck that switches language with `simulator lang=...` comes out as one
SystemVerilog file:

In [7]:
SPECTRE_SPICE = """\
simulator lang=spice
.subckt rdiv a b mid
R1 a mid 1k
R2 mid b 1k
.ends rdiv
simulator lang=spectre
subckt top (vin gnd)
X0 (vin gnd tap) rdiv
C0 (tap gnd) capacitor c=1p
ends top
"""
print(convert(SPECTRE_SPICE, "spectre_spice", "systemverilog"))

module rdiv (
  inout a,
  inout b,
  inout mid
);
  resistor #(.value(1000)) R1 (.p(a), .n(mid));
  resistor #(.value(1000)) R2 (.p(mid), .n(b));
endmodule

module top (
  inout vin,
  inout gnd
);
  rdiv X0 (.a(vin), .b(gnd), .mid(tap));
  capacitor #(.c(1e-12)) C0 (.p(tap), .n(gnd));
endmodule



## The conversion matrix

The same CMOS buffer, written by hand in each of the 7 formats:

In [8]:
SOURCES = {
    "spice": SPICE,
    "hspice": """\
* buffer
.subckt inverter in out vdd vss
M1 out in vdd vdd pch W=2u L=180n   $ pull-up
M2 out in vss vss nch W=1u L=180n   $ pull-down
.ends inverter
.subckt buffer a y vdd vss
X1 a mid vdd vss inverter
X2 mid y vdd vss inverter
C1 y vss 10f
.ends buffer
.end
""",
    "xyce": """\
.SUBCKT inverter in out vdd vss
M1 out in vdd vdd pch W=2u L=180n
M2 out in vss vss nch W=1u L=180n
.ENDS inverter
.SUBCKT buffer a y vdd vss
X1 a mid vdd vss inverter
X2 mid y vdd vss inverter
C1 y vss 10f
.ENDS buffer
.END
""",
    "spectre": """\
subckt inverter (in out vdd vss)
M1 (out in vdd vdd) pch W=2u L=180n
M2 (out in vss vss) nch W=1u L=180n
ends inverter
subckt buffer (a y vdd vss)
X1 (a mid vdd vss) inverter
X2 (mid y vdd vss) inverter
C1 (y vss) capacitor value=10f
ends buffer
""",
    "spectre_spice": """\
simulator lang=spice
.subckt inverter in out vdd vss
M1 out in vdd vdd pch W=2u L=180n
M2 out in vss vss nch W=1u L=180n
.ends inverter
simulator lang=spectre
subckt buffer (a y vdd vss)
X1 (a mid vdd vss) inverter
X2 (mid y vdd vss) inverter
C1 (y vss) capacitor value=10f
ends buffer
""",
    "vams": """\
module inverter (in, out, vdd, vss);
  inout in, out, vdd, vss;
  electrical in, out, vdd, vss;
  mosfet #(.W(2u), .L(180n)) M1 (.d(out), .g(in), .s(vdd), .b(vdd));
  mosfet #(.W(1u), .L(180n)) M2 (.d(out), .g(in), .s(vss), .b(vss));
endmodule
module buffer (a, y, vdd, vss);
  inout a, y, vdd, vss;
  electrical a, y, vdd, vss;
  inverter X1 (.in(a), .out(mid), .vdd(vdd), .vss(vss));
  inverter X2 (.in(mid), .out(y), .vdd(vdd), .vss(vss));
  capacitor #(.value(10f)) C1 (.p(y), .n(vss));
endmodule
""",
    "systemverilog": """\
module inverter (inout in, out, vdd, vss);
  mosfet #(.W(2e-6), .L(180e-9)) M1 (.d(out), .g(in), .s(vdd), .b(vdd));
  mosfet #(.W(1e-6), .L(180e-9)) M2 (.d(out), .g(in), .s(vss), .b(vss));
endmodule
module buffer (inout a, y, vdd, vss);
  inverter X1 (.in(a), .out(mid), .vdd, .vss);
  inverter X2 (.in(mid), .out(y), .vdd, .vss);
  capacitor #(.value(10e-15)) C1 (.p(y), .n(vss));
endmodule
""",
}

Two netlists are "the same" when every circuit has the same ports, the same
nets and the same instances with the same component and settings, as seen by
`kfnetlist.Netlist`:

In [9]:
def graph(netlist: morphnet.Netlist) -> dict:
    out = {}
    for name, nl in morphnet.to_netlists(netlist).items():
        nl.sort()
        nets = sorted(
            sorted(
                m.name if isinstance(m, NetlistPort) else f"{m.instance}.{m.port}"
                for m in net
            )
            for net in nl.nets
        )
        instances = {
            k: (v.component, {s: float(f"{x:.9g}") for s, x in v.settings.items()})
            for k, v in nl.instances.items()
        }
        out[name] = (sorted(p.name for p in nl.ports), nets, instances)
    return out


reference = graph(formats.parse_spice(SPICE))
for name, (ports, nets, instances) in reference.items():
    print(f"{name}: ports={ports}")
    print(f"  nets={nets}")
    print(f"  instances={instances}")

inverter: ports=['in', 'out', 'vdd', 'vss']
  nets=[['M1.b', 'M1.s', 'vdd'], ['M1.d', 'M2.d', 'out'], ['M1.g', 'M2.g', 'in'], ['M2.b', 'M2.s', 'vss']]
  instances={'M1': ('mosfet', {'L': 1.8e-07, 'W': 2e-06}), 'M2': ('mosfet', {'L': 1.8e-07, 'W': 1e-06})}
buffer: ports=['a', 'vdd', 'vss', 'y']
  nets=[['C1.n', 'X1.vss', 'X2.vss', 'vss'], ['C1.p', 'X2.out', 'y'], ['X1.in', 'a'], ['X1.out', 'X2.in'], ['X1.vdd', 'X2.vdd', 'vdd']]
  instances={'C1': ('capacitor', {'value': 1e-14}), 'X1': ('inverter', {}), 'X2': ('inverter', {})}


All 7 sources parse to that graph:

In [10]:
for src, text in SOURCES.items():
    parsed = getattr(formats, f"parse_{src}")(text)
    print(f"{src:>14}: {'same' if graph(parsed) == reference else 'DIFFERENT'}")

         spice: same
        hspice: same


          xyce: same
       spectre: same
 spectre_spice: same
          vams: same
 systemverilog: same


Now convert each source (rows) to each target format (columns), parse the
result back, and compare it with the reference:

In [11]:
width = max(map(len, FORMATS)) + 1
print(" " * width + "".join(f"{dst:>{width}}" for dst in FORMATS))
for src, text in SOURCES.items():
    row = []
    for dst in FORMATS:
        back = getattr(formats, f"parse_{dst}")(convert(text, src, dst))
        row.append("✓" if graph(back) == reference else "✗")
    print(f"{src:>{width}}" + "".join(f"{cell:>{width}}" for cell in row))

                       spice        hspice          xyce       spectre spectre_spice          vams systemverilog


         spice             ✓             ✓             ✓             ✓             ✓             ✓             ✓


        hspice             ✓             ✓             ✓             ✓             ✓             ✓             ✓


          xyce             ✓             ✓             ✓             ✓             ✓             ✓             ✓
       spectre             ✓             ✓             ✓             ✓             ✓             ✓             ✓


 spectre_spice             ✓             ✓             ✓             ✓             ✓             ✓             ✓
          vams             ✓             ✓             ✓             ✓             ✓             ✓             ✓


 systemverilog             ✓             ✓             ✓             ✓             ✓             ✓             ✓


Connectivity and parameter values survive all 49 conversions. Some
information is format-specific and does not:

| Information | Kept by | Lost when written to |
|---|---|---|
| Device model name (`pch`, `nch`) | SPICE, HSPICE, Xyce, Spectre, Spectre-SPICE | Verilog-AMS, SystemVerilog |
| `analog` blocks | Verilog-AMS | every other format |
| Behavioral blocks (`always`, `initial`, ...) | SystemVerilog | every other format |
| Port directions | Verilog-AMS, SystemVerilog | SPICE family and Spectre (all `inout`) |
| SI suffixes (`10k`) | all but SystemVerilog | SystemVerilog writes `10000` |
| Pin names of undefined cells | Verilog-AMS, SystemVerilog (named connections) | SPICE family and Spectre (positional) |

For example, the model names are gone after a trip through Verilog-AMS:

In [12]:
print(convert(convert(SPICE, "spice", "vams"), "vams", "spice"))

* untitled
.subckt inverter in out vdd vss
M1 out in vdd vdd W=2u L=180n
M2 out in vss vss W=1u L=180n
.ends inverter
.subckt buffer a y vdd vss
X1 a mid vdd vss inverter
X2 mid y vdd vss inverter
C1 y vss 10f
.ends buffer
.end



## A photonic MZI through every format

Terminals named `o1`, `o2`, ... are waveguide terminals in every format, so
a photonic circuit keeps its waveguide nets through any conversion. The leaf
cells are declared as empty modules so that every format knows their pin
names:

In [13]:
MZI = """\
module dc #(parameter real gap = 200e-9) (inout o1, o2, o3, o4);
endmodule
module wg #(parameter real length = 100e-6) (inout o1, o2);
endmodule
module heater_wg #(parameter real length = 100e-6) (inout o1, o2, e1, e2);
endmodule

module mzi (input o1, o2, output o3, o4, input e1, inout gnd);
  dc #(.gap(200e-9)) Xdc1 (.o1(o1), .o2(o2), .o3(top_a), .o4(bot_a));
  dc #(.gap(200e-9)) Xdc2 (.o1(top_b), .o2(bot_b), .o3(o3), .o4(o4));
  heater_wg Xtop (.o1(top_a), .o2(top_b), .e1(e1), .e2(gnd));
  wg Xbot (.o1(bot_a), .o2(bot_b));
endmodule
"""
mzi = formats.parse_systemverilog(MZI)


def net_domains(netlist: morphnet.Netlist) -> dict[str, str]:
    top = next(c for c in netlist.circuits if c.name == "mzi")
    return {c.name: c.domain.name.lower() for c in top.connections}


expected = net_domains(mzi)
print(expected)

{'o1': 'waveguide', 'o2': 'waveguide', 'o3': 'waveguide', 'o4': 'waveguide', 'e1': 'electrical', 'gnd': 'electrical', 'top_a': 'waveguide', 'bot_a': 'waveguide', 'top_b': 'waveguide', 'bot_b': 'waveguide'}


In [14]:
for dst in FORMATS:
    text = getattr(formats, f"write_{dst}")(mzi)
    back = getattr(formats, f"parse_{dst}")(text)
    assert net_domains(back) == expected, dst
    assert graph(back) == graph(mzi), dst
    print(
        f"{dst:>14}: waveguide nets {sorted(n for n, d in expected.items() if d == 'waveguide')}"
    )

         spice: waveguide nets ['bot_a', 'bot_b', 'o1', 'o2', 'o3', 'o4', 'top_a', 'top_b']
        hspice: waveguide nets ['bot_a', 'bot_b', 'o1', 'o2', 'o3', 'o4', 'top_a', 'top_b']
          xyce: waveguide nets ['bot_a', 'bot_b', 'o1', 'o2', 'o3', 'o4', 'top_a', 'top_b']
       spectre: waveguide nets ['bot_a', 'bot_b', 'o1', 'o2', 'o3', 'o4', 'top_a', 'top_b']
 spectre_spice: waveguide nets ['bot_a', 'bot_b', 'o1', 'o2', 'o3', 'o4', 'top_a', 'top_b']
          vams: waveguide nets ['bot_a', 'bot_b', 'o1', 'o2', 'o3', 'o4', 'top_a', 'top_b']
 systemverilog: waveguide nets ['bot_a', 'bot_b', 'o1', 'o2', 'o3', 'o4', 'top_a', 'top_b']


The MZI in Spectre and in Verilog-AMS:

In [15]:
print(formats.write_spectre(mzi))

subckt dc (o1 o2 o3 o4) gap=2e-07
ends dc
subckt wg (o1 o2) length=0.0001
ends wg
subckt heater_wg (o1 o2 e1 e2) length=0.0001
ends heater_wg
subckt mzi (o1 o2 o3 o4 e1 gnd)
Xdc1 (o1 o2 top_a bot_a) dc gap=2e-07
Xdc2 (top_b bot_b o3 o4) dc gap=2e-07
Xtop (top_a top_b e1 gnd) heater_wg
Xbot (bot_a bot_b) wg
ends mzi



In [16]:
print(formats.write_vams(mzi))

module dc (o1, o2, o3, o4);
  inout o1, o2, o3, o4;
  parameter real gap = 2e-07;
endmodule
module wg (o1, o2);
  inout o1, o2;
  parameter real length = 0.0001;
endmodule
module heater_wg (o1, o2, e1, e2);
  inout o1, o2, e1, e2;
  parameter real length = 0.0001;
endmodule
module mzi (o1, o2, o3, o4, e1, gnd);
  input o1, o2, e1;
  output o3, o4;
  inout gnd;
  dc #(.gap(2e-07)) Xdc1 (.o1(o1), .o2(o2), .o3(top_a), .o4(bot_a));
  dc #(.gap(2e-07)) Xdc2 (.o1(top_b), .o2(bot_b), .o3(o3), .o4(o4));
  heater_wg Xtop (.o1(top_a), .o2(top_b), .e1(e1), .e2(gnd));
  wg Xbot (.o1(bot_a), .o2(bot_b));
endmodule



## From `kfnetlist.Netlist` to any format and back

`morphnet.from_netlists()` turns `{cell name: kfnetlist.Netlist}` into a
morphnet `Netlist`. Instances of built-in devices (`resistor`,
`capacitor`, `mosfet`, ...) get the same pin declarations the SPICE parser
creates, so positional formats write their pins in device order.

In [17]:
rc = Netlist()
rc.create_inst("R1", kcl="", component="resistor", settings={"value": 1000.0})
rc.create_inst("C1", kcl="", component="capacitor", settings={"value": 1e-10})
for port in ("vin", "vout", "gnd"):
    rc.create_port(port)
rc.create_net(NetlistPort("vin"), PortRef("R1", "p"))
rc.create_net(PortRef("R1", "n"), PortRef("C1", "p"), NetlistPort("vout"))
rc.create_net(PortRef("C1", "n"), NetlistPort("gnd"))
rc.sort()

doc = morphnet.from_netlists({"rc": rc}, "rc")
for dst in FORMATS:
    text = getattr(formats, f"write_{dst}")(doc)
    back = getattr(formats, f"parse_{dst}")(text)
    print(f"{dst:>14}: {'same graph' if graph(back) == graph(doc) else 'DIFFERENT'}")

         spice: same graph


        hspice: same graph
          xyce: same graph
       spectre: same graph
 spectre_spice: same graph
          vams: same graph
 systemverilog: same graph


Ports, nets, components and settings come back unchanged. kfnetlist-only
instance data (instance arrays, placement) has no place in the text
formats; use `to_json()` / protobuf for a lossless copy.

In [18]:
print(formats.write_systemverilog(doc))

module rc (
  inout gnd,
  inout vin,
  inout vout
);
  capacitor #(.value(1e-10)) C1 (.p(vout), .n(gnd));
  resistor #(.value(1000)) R1 (.p(vin), .n(vout));
endmodule

